In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [8]:
import pandas as pd
import json

df = pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")

df = df[["State", "City", "Severity"]]
df = df.dropna()

# Count accidents for each State-City combination
grouped = df.groupby(["State", "City"]).size().reset_index(name="count")

tree = {
    "name": "US Accidents",
    "children": []
}

for state, state_data in grouped.groupby("State"):

    state_node = {
        "name": str(state),
        "children": []
    }

    for _, row in state_data.iterrows():

        city_node = {
            "name": str(row["City"]),
            "value": int(row["count"])
        }

        state_node["children"].append(city_node)

    tree["children"].append(state_node)

with open("accidents_state_city.json", "w") as f:
    json.dump(tree, f)

In [9]:
import pandas as pd
import json
import os

# Group accidents by State and Severity
grouped = (
    df.groupby(["State", "Severity"])
    .size()
    .reset_index(name="count")
)

tree = {
    "name": "US Accidents",
    "children": []
}

for state, state_data in grouped.groupby("State"):

    state_node = {
        "name": str(state),
        "children": []
    }

    for _, row in state_data.iterrows():

        severity_node = {
            "name": "Severity " + str(int(row["Severity"])),
            "value": int(row["count"])
        }

        state_node["children"].append(severity_node)

    tree["children"].append(state_node)

os.makedirs("data", exist_ok=True)

with open("data/accidents_state_severity.json", "w") as f:
    json.dump(tree, f)

In [11]:
import pandas as pd
import json
import os
df=pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")
df = df.dropna(subset=["Civil_Twilight", "Severity"]).copy()

df["Civil_Twilgiht"] = df["Civil_Twilight"].str.strip().str.title()

grouped = (
    df.groupby(["Civil_Twilight", "Severity"])
    .size()
    .reset_index(name="count")
)

periods = ["Day", "Night"]

tree = {
    "name": "US Accidents",
    "children": []
}

for period in periods:

    period_data = grouped[grouped["Civil_Twilight"] == period]

    period_node = {
        "name": period,
        "children": []
    }

    for _, row in period_data.iterrows():

        period_node["children"].append({
            "name": "Severity " + str(int(row["Severity"])),
            "value": int(row["count"])
        })

    tree["children"].append(period_node)

os.makedirs("data", exist_ok=True)

with open("data/accidents_day_night_severity.json", "w") as f:
    json.dump(tree, f)